# Exp 8: three frozen checks (run top to bottom)

**How to use:** after any kernel restart, use **Run → Run All Cells**. Everything below the red STOP line is guarded and will not start any GPU job unless you change a flag in STEP 2. The order is already correct: definitions → config → freeze → status → gate → checks.

| Step | What it does | Needs |
|---|---|---|
| 1a, 1b | defines the functions (prints nothing) | nothing |
| 2 | config and safety flags | nothing |
| 3 | **freeze** the criteria (once; it refuses to overwrite) | nothing. Must run before you read joint results |
| 4 | what files exist | nothing |
| 1c | **exact entropy floor** (v4: replaces the tree floor, which was too high because strings are ambiguous) | nothing |
| 5, 5a, 5b, 5c | **gate only** (exact-floor corrected), optional floor check, record amendments 1 and 2 | joint seed 0 |
| 6, 7 | Check 1 (dissociation), Check 2 (beats string distance) | 3 seeds of joint results for a verdict |
| 8 | noise in a(g) across translator seeds | translators |
| STOP | GPU cells below | `RUN_GPU = True` |

Between PASS and FAIL = INCONCLUSIVE. A failed gate means untestable at this d, not negative.

## STEP 1a. Definitions: checks 1 and 2

In [1]:
# Definitions for checks 1 and 2 (CPU only). Nothing runs here.
import os, sys, json, glob, random, time, subprocess
import numpy as np
import exp8_lib as X

# ----------------------------------------------------------------------------- frozen thresholds (edit only BEFORE joint results exist)
GATE_MIN_GAIN, GATE_Z = 0.02, 3.0        # same instrument gate as the notebook
C1_MARGIN        = 0.10                  # s(rev_all) - max(s(rev4), s(rev45)) must be >= this in EVERY seed
C1_MIN_SEEDS     = 3                     # fewer seeds -> no verdict
C2_MIN_N         = 8                     # renderings with finite a and s
C2_MIN_SEEDS     = 3                     # fewer seeds -> numbers are printed but there is no verdict
C2_PARTIAL_PASS  = -0.5                  # partial rho(a, s | tau, displacement) must be <= this
C2_PARTIAL_DEAD  = -0.2                  # ... and is DEAD if >= this
C2_LOO_MAX       = -0.3                  # leave-one-rendering-out partial must never exceed this
C2_BEAT_MARGIN   = 0.10                  # rho(a, s) must be at least this much more negative than rho(tau, s)

PREREG = f"""H-adapter pre-registration - written {{when}}
Written before ANY joint (s) result was read. Translator costs a(g) for seed 0 had already been seen.

GATE   mean gain_A' >= {GATE_MIN_GAIN} nats and >= {GATE_Z} SE. If it fails, every check below is UNTESTABLE, not negative.

CHECK 1 (dissociation) needs >= {C1_MIN_SEEDS} seeds.
  PASS         s(rev_all) - max(s(rev4), s(rev45)) >= {C1_MARGIN} in every seed
  FAIL         mean of that difference <= 0
  INCONCLUSIVE otherwise
  Secondary (reported, not part of the verdict): perm_sym is among the two lowest s in every seed.

CHECK 2 (a beats string distance) needs >= {C2_MIN_N} renderings with finite a and s, and >= {C2_MIN_SEEDS} seeds.
  PASS         partial rho(a, s | tau, displacement) <= {C2_PARTIAL_PASS}
               AND rho(a, s) <= rho(tau, s) - {C2_BEAT_MARGIN}
               AND the leave-one-rendering-out partial never exceeds {C2_LOO_MAX}
  FAIL         partial >= {C2_PARTIAL_DEAD}
  INCONCLUSIVE otherwise

CHECK 3 (depth) - see exp8_depth.py for its frozen thresholds.
"""

def write_prereg(root):
    path = os.path.join(root, "prereg_three_checks.txt")
    if os.path.exists(path): print("already frozen:", path); print(open(path).read()); return
    os.makedirs(root, exist_ok=True); open(path, "w").write(PREREG.format(when=time.strftime("%Y-%m-%d %H:%M"))); print("wrote", path)

AMENDMENT_1 = """AMENDMENT 1 ({when}). Found because the gate printed gain_A' = -0.0435 nats for every rendering (z about -30).
Cause: the disjoint control uses permuted grammars, whose per-token entropy floors differ from the shared grammar's (A' floor 0.7379 vs 0.7714 nats/token), so raw CE differences mix grammar entropy with sharing.
Change: every gain is now computed on excess CE over the analytic entropy floor, excess = CE - floor, on the same evaluation trees. Thresholds are unchanged.
This was decided from the gate (anchor language A') only, before any rendering's s was read."""

def amend_prereg(root, text=None):
    path = os.path.join(root, "prereg_three_checks.txt"); text = (text or AMENDMENT_1).format(when=time.strftime("%Y-%m-%d %H:%M"))
    if not os.path.exists(path): print("freeze first: run write_prereg"); return
    if text.split("(")[0] in open(path).read(): print("amendment already recorded"); return
    open(path, "a").write("\n" + text + "\n"); print("amendment appended to", path)

# ----------------------------------------------------------------------------- loading (same definitions as the notebook)
def load_all(root):
    J, T = {}, {}
    for p in glob.glob(os.path.join(root, "joint", "*.json")):
        r = json.load(open(p)); J[(r["g"], r["cond"], r["d"], r["seed"])] = r
    for p in glob.glob(os.path.join(root, "translator", "*.json")):
        r = json.load(open(p)); T[(r["g"], r["d"], r["seed"])] = r
    return J, T

_FLOOR = {}
def _nll(rules, node):
    if isinstance(node, int): return 0.0
    sym, ch = node
    return float(np.log(len(rules[sym]))) + sum(_nll(rules, c) for c in ch)

def floor_ce(cond, lang, g, n=1000, seed=424242, max_leaves=160):
    """Entropy floor (nats per token) of language `lang` in condition `cond`, on exactly the trees the joint evaluation uses (seed 424242, <= 160 leaves).
    Every rendering is injective, so the tokens carry the tree's full information and the best possible per-token CE is nll(tree) / len(rendering).
    The permuted (disjoint) grammars have DIFFERENT entropies from the shared grammar, so raw CE cannot be compared across conditions."""
    key = (cond, lang, g if lang == "G" else "-")
    if key not in _FLOOR:
        gram = X.make_grammars(cond)[lang]; spec = X.TRANSFORMS[g] if lang == "G" else {}; rng = random.Random(seed); v = []
        while len(v) < n:
            t = X.sample_tree(gram, rng)
            if len(X.render(t, {})) > max_leaves: continue
            v.append(_nll(gram, t) / len(X.render(t, spec)))
        _FLOOR[key] = float(np.mean(v))
    return _FLOOR[key]

def gains(J, d, seeds, use_floor=True):
    """gain_L = excess_disjoint(L) - excess_shared(L), with excess = CE - entropy floor. use_floor=False reproduces the raw CE difference (confounded by grammar entropy)."""
    out = {}
    for g in X.G_LIST:
        for s in seeds:
            a, b = J.get((g, "shared", d, s)), J.get((g, "disjoint", d, s))
            if a is None or b is None: continue
            fl = (lambda cond, L: floor_ce(cond, L, g)) if use_floor else (lambda cond, L: 0.0)
            ex_sh = {L: a["ce"][L] - fl("shared", L) for L in X.LANGS3}; ex_dj = {L: b["ce"][L] - fl("disjoint", L) for L in X.LANGS3}
            gain = {L: ex_dj[L] - ex_sh[L] for L in X.LANGS3}
            se = {L: (a["ce_se"][L] ** 2 + b["ce_se"][L] ** 2) ** 0.5 for L in X.LANGS3}
            ok = gain["A2"] > max(GATE_MIN_GAIN, GATE_Z * se["A2"])
            out[(g, s)] = dict(gain=gain, se=se, anchor_ok=ok, s=gain["G"] / gain["A2"] if ok else float("nan"), excess_shared=ex_sh, excess_disjoint=ex_dj)
    return out

def cost_a(T, d, seeds):
    out = {}
    for g in X.G_LIST:
        v = [T[(g, d, s)]["auc"] - T[("identity", d, s)]["auc"] for s in seeds if (g, d, s) in T and ("identity", d, s) in T]
        if v: out[g] = float(np.mean(v))
    return out

def gate_status(G):
    if not G: return False, "no joint runs yet"
    anchors = [v["gain"]["A2"] for v in G.values()]; ses = [v["se"]["A2"] for v in G.values()]
    m = float(np.mean(anchors)); sem = max(float(np.mean(ses)) / len(anchors) ** 0.5, 1e-9)
    ok = m >= GATE_MIN_GAIN and m >= GATE_Z * sem
    return ok, f"mean gain_A' {m:.4f} nats over {len(anchors)} runs (SE of mean ~{sem:.4f})"

# ----------------------------------------------------------------------------- stats helpers
def _ranks(x):
    x = np.asarray(x, float); order = np.argsort(x); r = np.empty(len(x)); i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]: j += 1
        r[order[i:j + 1]] = (i + j) / 2 + 1; i = j + 1
    return r

def spearman(x, y):
    rx, ry = _ranks(x), _ranks(y)
    if rx.std() < 1e-12 or ry.std() < 1e-12: return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])

def _resid(v, Zr): return v - Zr @ np.linalg.lstsq(Zr, v, rcond=None)[0]

def partial_rank_corr(x, y, Z):
    """Spearman of x and y after regressing both ranks on the ranks of every column in Z (Z: list of vectors)."""
    n = len(x); Zr = np.column_stack([np.ones(n)] + [_ranks(z) for z in Z])
    ex, ey = _resid(_ranks(x), Zr), _resid(_ranks(y), Zr)
    if ex.std() < 1e-12 or ey.std() < 1e-12: return float("nan")
    return float(np.corrcoef(ex, ey)[0, 1])

def perm_p_partial(x, y, Z, obs, n=20000, seed=0):
    """one-sided permutation p for partial <= obs, permuting the residual of y (approximate; n is tiny, read it as a rough guide)."""
    rng = np.random.RandomState(seed); m = len(x); Zr = np.column_stack([np.ones(m)] + [_ranks(z) for z in Z])
    ex, ey = _resid(_ranks(x), Zr), _resid(_ranks(y), Zr)
    return float(np.mean([np.corrcoef(ex, rng.permutation(ey))[0, 1] <= obs for _ in range(n)]))

# ----------------------------------------------------------------------------- string baselines (same as the notebook cell)
def _uniq(node, c):
    if isinstance(node, int): c[0] += 1; return 1000 + c[0] - 1
    return (node[0], [_uniq(ch, c) for ch in node[1]])

def string_baselines(g, n_trees=300, seed=3):
    rules, _ = X.build_grammar(X.GRAMMAR_SEED); rng = random.Random(seed); spec = X.TRANSFORMS[g]; disp, tau = [], []
    for _ in range(n_trees):
        t = _uniq(X.sample_tree(rules, rng), [0]); out = X.render(t, spec)
        leaves = np.array([x - 1000 for x in out if x >= 1000]); n = len(leaves); pos = np.empty(n, int); pos[leaves] = np.arange(n)
        disp.append(np.abs(pos - np.arange(n)).mean() / n)
        tau.append(((pos[:, None] > pos[None, :]) & (np.arange(n)[:, None] < np.arange(n)[None, :])).sum() / (n * (n - 1) / 2))
    return dict(displacement=float(np.mean(disp)), kendall_tau=float(np.mean(tau)))

# ----------------------------------------------------------------------------- CHECK 1
def check1(root, d=32, seeds=(0, 7, 13), verbose=True):
    J, T = load_all(root); G = gains(J, d, list(seeds)); ok, msg = gate_status(G)
    print(f"=== CHECK 1: rev_all vs rev4 / rev45  (d={d}) ===\nGATE: {msg} -> {'PASS' if ok else 'FAIL'}")
    if not ok: print("VERDICT: UNTESTABLE (gate). Not a negative result. Try smaller d."); return dict(verdict="UNTESTABLE")
    need = ("rev_all", "rev4", "rev45"); diffs = []; sec = []
    print(f"\n{'seed':>5s} {'s(rev_all)':>10s} {'s(rev4)':>8s} {'s(rev45)':>9s} {'diff':>7s}   perm_sym s / rank among all")
    for s in seeds:
        v = {g: G[(g, s)]["s"] for g in X.G_LIST if (g, s) in G}
        if not all(g in v and np.isfinite(v[g]) for g in need): print(f"{s:>5d}   missing or gated-out runs"); continue
        diff = v["rev_all"] - max(v["rev4"], v["rev45"]); diffs.append(diff)
        fin = sorted([(x, g) for g, x in v.items() if np.isfinite(x)]); low2 = [g for _, g in fin[:2]]
        sec.append("perm_sym" in low2)
        print(f"{s:>5d} {v['rev_all']:10.3f} {v['rev4']:8.3f} {v['rev45']:9.3f} {diff:+7.3f}   {v.get('perm_sym', float('nan')):.3f} / {[g for _, g in fin].index('perm_sym') + 1 if 'perm_sym' in v and np.isfinite(v['perm_sym']) else '-'} of {len(fin)}")
    if len(diffs) < C1_MIN_SEEDS:
        print(f"\nVERDICT: NO VERDICT YET - {len(diffs)} usable seed(s), need {C1_MIN_SEEDS}. (Diffs so far: {[round(x, 3) for x in diffs]}.)"); return dict(verdict="NO VERDICT", diffs=diffs)
    if all(x >= C1_MARGIN for x in diffs): v = "PASS"
    elif float(np.mean(diffs)) <= 0: v = "FAIL"
    else: v = "INCONCLUSIVE"
    print(f"\nmean diff {np.mean(diffs):+.3f}, min {min(diffs):+.3f}, max {max(diffs):+.3f}   |   perm_sym among two lowest in {sum(sec)}/{len(sec)} seeds (secondary)")
    print(f"VERDICT: {v}" + ("  <- the dissociation holds; a(g) is not just string distance" if v == "PASS" else "  <- rev_all does NOT share better; a(g) loses its main argument" if v == "FAIL" else ""))
    return dict(verdict=v, diffs=diffs)

# ----------------------------------------------------------------------------- CHECK 2
def check2(root, d=32, seeds=(0, 7, 13), n_boot=2000):
    J, T = load_all(root); G = gains(J, d, list(seeds)); A = cost_a(T, d, list(seeds)); ok, msg = gate_status(G)
    print(f"=== CHECK 2: does a(g) beat string distance?  (d={d}, seeds={list(seeds)}) ===\nGATE: {msg} -> {'PASS' if ok else 'FAIL'}")
    if not ok: print("VERDICT: UNTESTABLE (gate)."); return dict(verdict="UNTESTABLE")
    gs, a, s = [], [], []
    for g in X.G_LIST:
        v = [G[(g, sd)]["s"] for sd in seeds if (g, sd) in G]; v = [x for x in v if np.isfinite(x)]
        if g in A and v: gs.append(g); a.append(A[g]); s.append(float(np.mean(v)))
    if len(gs) < C2_MIN_N: print(f"VERDICT: NO VERDICT YET - {len(gs)} renderings with finite a and s, need {C2_MIN_N}."); return dict(verdict="NO VERDICT")
    B = {g: string_baselines(g) for g in gs}; tau = [B[g]["kendall_tau"] for g in gs]; disp = [B[g]["displacement"] for g in gs]
    r_as, r_ts, r_ds = spearman(a, s), spearman(tau, s), spearman(disp, s)
    p_both = partial_rank_corr(a, s, [tau, disp]); p_tau = partial_rank_corr(a, s, [tau]); p_disp = partial_rank_corr(a, s, [disp])
    loo = [partial_rank_corr([x for j, x in enumerate(a) if j != i], [x for j, x in enumerate(s) if j != i], [[x for j, x in enumerate(z) if j != i] for z in (tau, disp)]) for i in range(len(gs))]
    rng = np.random.RandomState(1); boots = []
    for _ in range(n_boot):
        ix = rng.randint(0, len(gs), len(gs)); b = partial_rank_corr(np.array(a)[ix], np.array(s)[ix], [np.array(tau)[ix], np.array(disp)[ix]])
        if np.isfinite(b): boots.append(b)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (float("nan"),) * 2)
    print(f"\nn = {len(gs)} renderings: {gs}")
    print(f"rho(a, s)               = {r_as:+.2f}")
    print(f"rho(tau, s)             = {r_ts:+.2f}     rho(displacement, s) = {r_ds:+.2f}   <- the baselines")
    print(f"partial rho(a,s | tau)  = {p_tau:+.2f}     | disp = {p_disp:+.2f}     | tau AND disp = {p_both:+.2f}   (bootstrap 95% CI over renderings [{lo:+.2f}, {hi:+.2f}])")
    print(f"leave-one-out partial   range [{np.nanmin(loo):+.2f}, {np.nanmax(loo):+.2f}]   perm p (partial <= observed) = {perm_p_partial(a, s, [tau, disp], p_both):.3f}")
    beats = r_as <= r_ts - C2_BEAT_MARGIN
    n_seeds = len({sd for (_, sd) in G})
    if n_seeds < C2_MIN_SEEDS: print(f"\nVERDICT: NO VERDICT YET - {n_seeds} seed(s) of joint results, need {C2_MIN_SEEDS}. The numbers above are provisional."); return dict(verdict="NO VERDICT", rho_as=r_as, partial=p_both)
    if p_both <= C2_PARTIAL_PASS and beats and np.nanmax(loo) <= C2_LOO_MAX: v = "PASS"
    elif p_both >= C2_PARTIAL_DEAD: v = "FAIL"
    else: v = "INCONCLUSIVE"
    print(f"a beats tau as a predictor of s by >= {C2_BEAT_MARGIN}: {beats}")
    print(f"VERDICT: {v}" + ("  <- a(g) carries information string distance does not" if v == "PASS" else "  <- once tau/displacement are controlled, a(g) predicts nothing" if v == "FAIL" else "  <- more renderings (rev3, rev345, rev2, rev23) and seeds before concluding"))
    print("Note: with ~10 renderings the partial has ~7 degrees of freedom; the bootstrap CI above is wide on purpose. Report it.")
    return dict(verdict=v, rho_as=r_as, rho_ts=r_ts, partial=p_both, ci=(float(lo), float(hi)))


## STEP 1c. Exact entropy floor (replaces the tree-entropy floor from 1a)
Runs on CPU, prints nothing but one line. Must sit after 1a because it overrides `floor_ce`.

In [2]:
import math
# Exact entropy floor, -log P(string) per token (CPU only). Replaces floor_ce from STEP 1a.
# Why: the model sees only the string, and a string can have several derivations, so the old floor nll(tree)/tokens was too high
# (by 0.0069 for the shared grammar and 0.0118 for the disjoint A' grammar). Each symbol c gets a matrix M_c[i, j] = P(c yields x[i:j]) in the
# RENDERED order; a rule is the product of its children's matrices (the matrix product sums over split points).
def _order(lvl, idx, n, spec):
    order = list(range(n))
    if lvl in spec.get("rev", ()): order = order[::-1]
    elif lvl in spec.get("swap", ()) and n >= 2: order[0], order[1] = order[1], order[0]
    elif "perm_sym" in spec: order = spec["perm_sym"][((lvl, idx), n)]
    return order

def string_logp(x, rules, sbl, spec):
    x = list(x); n = len(x); L = X.LEVELS; tm = spec.get("term_map", {}); mk = spec.get("marker"); M = {}
    for sym in sbl[L]:
        m = np.zeros((n + 1, n + 1)); deg = len(rules[sym])
        for rhs in rules[sym]:
            order = _order(L, sym[1], len(rhs), spec)
            r = [tm.get(rhs[j], rhs[j]) for j in order] + ([mk] if mk is not None else []); k = len(r)
            for i in range(n - k + 1):
                if x[i:i + k] == r: m[i, i + k] += 1.0 / deg
        M[sym] = m
    for lvl in range(L - 1, 0, -1):
        new = {}
        for sym in sbl[lvl]:
            deg = len(rules[sym]); acc = np.zeros((n + 1, n + 1))
            for rhs in rules[sym]:
                order = _order(lvl, sym[1], len(rhs), spec); P = M[rhs[order[0]]]
                for j in order[1:]: P = P @ M[rhs[j]]
                acc += P / deg
            new[sym] = acc
        M = new
    p = M[(1, 0)][0, n]
    return math.log(p) if p > 0 else float("-inf")

def eval_trees(rules, n_trees=1000, seed=424242, max_leaves=160):      # same trees as X.make_eval_rows
    rng = random.Random(seed); out = []
    while len(out) < n_trees:
        t = X.sample_tree(rules, rng)
        if len(X.render(t, {})) <= max_leaves: out.append(t)
    return out

_FLOOR.clear(); _SBL = X.build_grammar(X.GRAMMAR_SEED)[1]
import multiprocessing as mp
N_PROC = 1          # 1 = safe default (forking inside a Jupyter kernel can hang); raise to e.g. 16 once you have confirmed it does not hang for you
def _chunk(args):
    trees, gram, spec = args
    return [-string_logp(X.render(t, spec), gram, _SBL, spec) / len(X.render(t, spec)) for t in trees]
def floor_ce(cond, lang, g, n=1000, seed=424242, max_leaves=160):
    """Exact floor (nats per token) for language `lang` in condition `cond`; for lang G the rendering g is applied. Cached."""
    same = "canon" if (cond == "shared" and lang in ("A", "A2")) else lang          # shared A and A' are the same grammar, same trees, same rendering
    key = (cond, same, g if lang == "G" else "-")
    if key not in _FLOOR:
        gram = X.make_grammars(cond)[lang]; spec = X.TRANSFORMS[g] if lang == "G" else {}; trees = eval_trees(gram, n, seed, max_leaves)
        if N_PROC > 1:
            with mp.get_context("fork").Pool(N_PROC) as pool: parts = pool.map(_chunk, [(trees[i::N_PROC], gram, spec) for i in range(N_PROC)])
        else: parts = [_chunk((trees, gram, spec))]
        _FLOOR[key] = float(np.mean([v for q in parts for v in q]))
    return _FLOOR[key]
print(f"exact floor_ce installed (replaces the tree-entropy one), {N_PROC} processes. 24 floors are needed for STEP 5 (about 1-2 s each with many cores, ~20 s each with one).")

exact floor_ce installed (replaces the tree-entropy one), 1 processes. 24 floors are needed for STEP 5 (about 1-2 s each with many cores, ~20 s each with one).


## STEP 1b. Definitions: check 3 (depth)

In [3]:
"""exp8_depth.py - Check 3: does cross-rendering sharing start LATER in depth as conversion cost a(g) rises?

Readout (a cheap proxy for the Exp 7 patching; it is a probe, not a causal intervention):
  every token of a rendering has a ground-truth ancestor symbol at each tree level, whatever order the tokens are written in.
  Train a linear probe for the level-k ancestor on canonical-language (A) residual states at layer l, apply it unchanged to rendering-g states.
  T_l(g) = (acc[A-probe on g] - chance) / (acc[g-probe on g] - chance)
  The denominator is g's OWN ceiling (probe trained on g), not A's: reordered strings put different information in each prefix, so A's accuracy is not g's ceiling.
  onset(g) = first layer from which T stays >= ONSET_FRAC to the last layer; NaN if never.

Frozen prediction (edit only before the first run):
  Primary: PRIMARY_LEVEL = 4, centered probes, median over seeds.
  PASS  Spearman(a, onset) >= +0.7 over renderings with a defined onset, AND partial given tau >= +0.5
  FAIL  Spearman(a, onset) <= +0.3
  INCONCLUSIVE otherwise.  UNTESTABLE if fewer than 6 renderings reach ONSET_FRAC, or if the shared-model gate has not passed.
  Reported but not part of the verdict: other levels, strict (uncentered) probes, mean T over layers, an untrained-model floor.
"""
import os, sys, json, random, argparse
import numpy as np
import exp8_lib as X

LEVELS = X.LEVELS
PROBE_LEVELS = [2, 3, 4, 5, 6]          # level 1 has a single symbol, so nothing to decode
PRIMARY_LEVEL = 4
ONSET_FRAC = 0.5
MIN_CEIL_GAP = 0.05                     # layers where the g-probe barely beats chance on g are masked (T undefined)
N_TRAIN, N_TEST = 400, 400              # trees per language per split
TRAIN_SEED, TEST_SEED = 31, 32
C3_PASS_RHO, C3_PASS_PARTIAL, C3_FAIL_RHO, C3_MIN_DEFINED = 0.7, 0.5, 0.3, 6

# ----------------------------------------------------------------------------- rows with ancestor labels (torch-free)
def render_lab(node, spec, anc=()):
    """Mirror of X.render that also returns, per output token, the ancestor symbol index at levels 1..LEVELS.
    A terminal's ancestors are its LEVELS enclosing nonterminals; the marker token belongs to the level-LEVELS node that emitted it."""
    if isinstance(node, int): return [spec.get("term_map", {}).get(node, node)], [anc]
    (lvl, idx), children = node; anc2 = anc + (idx,)
    order = list(range(len(children)))
    if lvl in spec.get("rev", ()): order = order[::-1]
    elif lvl in spec.get("swap", ()) and len(order) >= 2: order[0], order[1] = order[1], order[0]
    elif "perm_sym" in spec: order = spec["perm_sym"][((lvl, idx), len(children))]
    toks, labs = [], []
    for j in order:
        t, l = render_lab(children[j], spec, anc2); toks.extend(t); labs.extend(l)
    if spec.get("marker") is not None and lvl == LEVELS: toks.append(spec["marker"]); labs.append(anc2)
    return toks, labs

def make_rows_lab(grammar, spec, bos, n_trees, seed, max_leaves=160, row_len=X.SEQ_LEN):
    """Same tree sampling as X.make_eval_rows (so the same seed gives the same trees in every language), plus labels [n, row_len, LEVELS] (-1 = no label)."""
    rng = random.Random(seed); rows = np.zeros((n_trees, row_len), np.int16); nreal = np.zeros(n_trees, np.int64); labs = -np.ones((n_trees, row_len, LEVELS), np.int8); k = 0
    while k < n_trees:
        t = X.sample_tree(grammar, rng)
        if len(X.render(t, {})) > max_leaves: continue
        toks, lab = render_lab(t, spec); assert toks == X.render(t, spec), "render_lab diverged from X.render"
        row = [bos] + toks; assert len(row) <= row_len
        rows[k, :len(row)] = row; nreal[k] = len(row); labs[k, 1:len(row)] = np.asarray(lab, np.int8); k += 1
    return rows, nreal, labs

# ----------------------------------------------------------------------------- probes (numpy)
def _ridge_fit(Xtr, y, ncls, lam=1e-2):
    Xb = np.concatenate([Xtr, np.ones((len(Xtr), 1))], 1); Y = np.eye(ncls)[y]
    return np.linalg.solve(Xb.T @ Xb + lam * len(Xb) * np.eye(Xb.shape[1]), Xb.T @ Y)

def _ridge_acc(W, Xte, y): return float((np.argmax(np.concatenate([Xte, np.ones((len(Xte), 1))], 1) @ W, 1) == y).mean())

def probe_transfer(S, Y, ncls=4, centered=True):
    """S[lang][split]: array [L, N, d]; Y[lang][split]: int array [N] (languages 'A', 'A2', 'G'; splits 'train', 'test').
    Returns per-layer dict: acc_AA, acc_AA2, acc_AG, acc_GG, chance, T (NaN where g's own ceiling is within MIN_CEIL_GAP of chance)."""
    L = S["A"]["train"].shape[0]; out = {k: [] for k in ("acc_AA", "acc_AA2", "acc_AG", "acc_GG", "chance", "T")}
    chance = float(np.bincount(Y["G"]["test"], minlength=ncls).max() / len(Y["G"]["test"]))
    for l in range(L):
        mu = {lg: S[lg]["train"][l].mean(0) for lg in S} if centered else {lg: S["A"]["train"][l].mean(0) for lg in S}
        f = lambda lg, sp: S[lg][sp][l] - mu[lg]
        WA = _ridge_fit(f("A", "train"), Y["A"]["train"], ncls); WG = _ridge_fit(f("G", "train"), Y["G"]["train"], ncls)
        aa, aa2 = _ridge_acc(WA, f("A", "test"), Y["A"]["test"]), _ridge_acc(WA, f("A2", "test"), Y["A2"]["test"])
        ag, gg = _ridge_acc(WA, f("G", "test"), Y["G"]["test"]), _ridge_acc(WG, f("G", "test"), Y["G"]["test"])
        T = (ag - chance) / (gg - chance) if gg - chance >= MIN_CEIL_GAP else float("nan")
        for k, v in zip(out, (aa, aa2, ag, gg, chance, T)): out[k].append(float(v))
    return out

def onset_layer(T, frac=ONSET_FRAC):
    """start of the final unbroken run of layers with T >= frac that reaches the last layer. A layer where T is undefined (g's own probe cannot decode yet)
    breaks the run: 'cannot be measured' is not evidence of sharing. NaN if the last layer is below frac or undefined."""
    T = np.asarray(T, float)
    if not np.isfinite(T[-1]) or T[-1] < frac: return float("nan")
    l = len(T) - 1
    while l - 1 >= 0 and np.isfinite(T[l - 1]) and T[l - 1] >= frac: l -= 1
    return float(l)

def decodable_layer(acc_GG, chance, gap=MIN_CEIL_GAP):
    """first layer where g's OWN probe beats chance by `gap`; onset cannot precede it, so onset - decodable is the lag attributable to transfer"""
    a = np.asarray(acc_GG, float) - chance; idx = np.where(a >= gap)[0]
    return float(idx[0]) if len(idx) else float("nan")

# ----------------------------------------------------------------------------- torch part: hidden states at every layer
try:
    import torch
    TORCH = True
except Exception:
    TORCH = False

if TORCH:
    @torch.no_grad()
    def layer_states(model, rows, nreal, labs, dev, bs=64):
        """returns states [L+1, Ntok, d] (embedding output, then every block output) at the rendering tokens (positions 1..nreal-1) and labels [Ntok, LEVELS]"""
        model.eval(); store = []; hs = [model.tok_emb.register_forward_hook(lambda m, i, o: store.append(o))]
        for blk in model.blocks: hs.append(blk.register_forward_hook(lambda m, i, o: store.append(o)))
        S, Ys = [], []; ar = torch.arange(rows.shape[1], device=dev)
        try:
            for i in range(0, rows.shape[0], bs):
                x = torch.from_numpy(rows[i:i + bs].astype(np.int64)).to(dev); nr = torch.from_numpy(nreal[i:i + bs]).to(dev); store.clear(); model(x)
                H = torch.stack(store, 0).float()                                   # [L+1, B, T, d]
                m = (ar[None, :] >= 1) & (ar[None, :] < nr[:, None])               # rendering tokens only, BOS and padding excluded
                S.append(H[:, m].cpu().numpy()); Ys.append(labs[i:i + bs][m.cpu().numpy()])
        finally:
            for h in hs: h.remove()
        return np.concatenate(S, 1), np.concatenate(Ys, 0)

    def load_model(root, g, cond, d, seed, dev, n_head=4, n_layer=12, untrained=False):
        if untrained: torch.manual_seed(seed + 1000)
        model = X.TinyRotaryGPT(X.VOCAB, d, n_layer, n_head, X.SEQ_LEN + 1).to(dev)
        if not untrained: model.load_state_dict(torch.load(X.joint_paths(root, g, cond, d, seed)[1], map_location=dev)["model_state"])
        return model

    def run_one(root, g, d, seed, device, untrained=False):
        dev = torch.device(device); path = os.path.join(root, "depth", f"{g}{'_untrained' if untrained else ''}_d{d}_s{seed}.json")
        if os.path.exists(path): print("done already:", path); return
        model = load_model(root, g, "shared", d, seed, dev, untrained=untrained); grams = X.make_grammars("shared"); specs = {"A": {}, "A2": {}, "G": X.TRANSFORMS[g]}
        S, Y = {}, {}
        for lg in ("A", "A2", "G"):
            S[lg], Y[lg] = {}, {}
            for sp, n, sd in (("train", N_TRAIN, TRAIN_SEED), ("test", N_TEST, TEST_SEED)):
                rows, nreal, labs = make_rows_lab(grams[lg], specs[lg], X.BOS[lg], n, sd); S[lg][sp], Y[lg][sp] = layer_states(model, rows, nreal, labs, dev)
        res = dict(g=g, d=d, seed=seed, untrained=untrained, levels={})
        for k in PROBE_LEVELS:
            Yk = {lg: {sp: Y[lg][sp][:, k - 1].astype(int) for sp in Y[lg]} for lg in Y}
            for centered in (True, False):
                r = probe_transfer(S, Yk, centered=centered); r["onset"] = onset_layer(r["T"]); res["levels"][f"{k}_{'centered' if centered else 'strict'}"] = r
        os.makedirs(os.path.dirname(path), exist_ok=True); json.dump(res, open(path, "w")); print("wrote", path, "level-4 centered T by layer:", np.round(res["levels"]["4_centered"]["T"], 2).tolist(), flush=True)

# ----------------------------------------------------------------------------- summary and verdict (numpy only)
def summarize(root, d=32, seeds=(0,), level=PRIMARY_LEVEL, mode="centered"):
    import glob
    key = f"{level}_{mode}"; per = {}
    for g in X.G_LIST:
        runs = [json.load(open(p)) for s in seeds for p in glob.glob(os.path.join(root, "depth", f"{g}_d{d}_s{s}.json"))]
        if runs: per[g] = dict(onset=[r["levels"][key]["onset"] for r in runs], T=np.nanmean([r["levels"][key]["T"] for r in runs], 0), acc_AA2=np.mean([r["levels"][key]["acc_AA2"] for r in runs], 0),
                      dec=decodable_layer(np.mean([r["levels"][key]["acc_GG"] for r in runs], 0), runs[0]["levels"][key]["chance"][0]))
    _, T = load_all(root); A = cost_a(T, d, list(seeds)); gs = [g for g in per if g in A]
    print(f"=== CHECK 3: depth of sharing (level {level}, {mode} probes, d={d}, seeds={list(seeds)}) ===")
    if len(gs) < 4: print("VERDICT: NO VERDICT YET - fewer than 4 renderings have both a(g) and a depth result."); return
    print(f"\n{'g':13s} {'a':>7s} {'onset':>6s} {'decod.':>6s} {'mean T':>7s}  T by layer 0..{len(per[gs[0]]['T']) - 1}   (decod. = first layer where g's own probe works)")
    for g in sorted(gs, key=lambda g: A[g]):
        on = float(np.nanmedian(per[g]["onset"])) if np.isfinite(per[g]["onset"]).any() else float("nan")
        print(f"{g:13s} {A[g]:7.4f} {on:6.1f} {per[g]['dec']:6.1f} {np.nanmean(per[g]['T']):7.2f}  " + " ".join("  . " if not np.isfinite(t) else f"{t:4.1f}" for t in per[g]["T"]))
        per[g]["on"] = on
    ok = [g for g in gs if np.isfinite(per[g]["on"])]; sanity = np.mean([per[g]["acc_AA2"][-1] for g in gs])
    print(f"\nsanity: A-probe on A' (same order, new tag) at the last layer = {sanity:.2f} (should be near the A-on-A accuracy)")
    if len(ok) < C3_MIN_DEFINED: print(f"VERDICT: UNTESTABLE - only {len(ok)} renderings reach T >= {ONSET_FRAC}; fewer than {C3_MIN_DEFINED}. Look at mean T instead, and at the shared-model gate."); return
    a = [A[g] for g in ok]; on = [per[g]["on"] for g in ok]; tau = [string_baselines(g)["kendall_tau"] for g in ok]
    rho, prt = spearman(a, on), partial_rank_corr(a, on, [tau]); rmt = spearman([A[g] for g in gs], [-np.nanmean(per[g]["T"]) for g in gs])
    lag = [per[g]["on"] - per[g]["dec"] for g in ok]; rlag = spearman(a, lag) if len(ok) >= 4 else float("nan")
    print(f"Spearman(a, onset) = {rho:+.2f} (n={len(ok)});  partial given tau = {prt:+.2f};  secondary: Spearman(a, -mean T) = {rmt:+.2f} (n={len(gs)}),  Spearman(a, onset - decodable layer) = {rlag:+.2f}")
    v = "PASS" if (rho >= C3_PASS_RHO and prt >= C3_PASS_PARTIAL) else "FAIL" if rho <= C3_FAIL_RHO else "INCONCLUSIVE"
    print(f"VERDICT: {v}" + ("  <- sharing starts later for costlier renderings, beyond string distance" if v == "PASS" else "  <- depth of sharing does not track a(g)" if v == "FAIL" else ""))
    print("Reminder: this is probe transfer (shared readout), not causal patching. A PASS motivates the patching version; it does not replace it.")


## STEP 2. Config and flags

In [4]:
ROOT, D_MODEL = "./exp8", 32
SEEDS = [0, 7, 13]            # checks 1 and 2 need >= 3 seeds of JOINT results for a verdict
DEVICE = "cuda:0"             # for the depth run only
RUN_GPU = False               # True -> the depth cells below will run on DEVICE
LAUNCH_JOINT_SEEDS = False    # True -> the cell near the bottom starts joint seeds 7 and 13 (only after the gate passes)
LAUNCH_PILOT = False          # True -> STEP 10a starts 8 small pilot jobs (marker and rev4, shared+disjoint, d=16 and d=8)
PILOT_DEVICE = "cuda:2"        # check nvidia-smi first
print("renderings:", X.G_LIST)
RECORD_AMENDMENT_2 = False   # True -> STEP 5c writes Amendment 2 into the frozen prereg file (set it only after STEP 5 shows no negative excess)


renderings: ['relabel_new', 'relabel_perm', 'marker', 'swap5', 'rev6', 'rev5', 'rev4', 'rev45', 'rev_all', 'perm_sym']


## STEP 3. Freeze the criteria (run once, before reading any joint result; safe to re-run)

In [5]:
write_prereg(ROOT)

already frozen: ./exp8/prereg_three_checks.txt
H-adapter pre-registration - written 2026-10-29 18:26
Written before ANY joint (s) result was read. Translator costs a(g) for seed 0 had already been seen.

GATE   mean gain_A' >= 0.02 nats and >= 3.0 SE. If it fails, every check below is UNTESTABLE, not negative.

CHECK 1 (dissociation) needs >= 3 seeds.
  PASS         s(rev_all) - max(s(rev4), s(rev45)) >= 0.1 in every seed
  FAIL         mean of that difference <= 0
  INCONCLUSIVE otherwise
  Secondary (reported, not part of the verdict): perm_sym is among the two lowest s in every seed.

CHECK 2 (a beats string distance) needs >= 8 renderings with finite a and s, and >= 3 seeds.
  PASS         partial rho(a, s | tau, displacement) <= -0.5
               AND rho(a, s) <= rho(tau, s) - 0.1
               AND the leave-one-rendering-out partial never exceeds -0.3
  FAIL         partial >= -0.2
  INCONCLUSIVE otherwise

CHECK 3 (depth) - see exp8_depth.py for its frozen thresholds.

AMENDM

## STEP 4. What exists so far

In [6]:
J, TR = load_all(ROOT)
print("T = translator, S/D = shared/disjoint joint, Y = shared checkpoint on disk (needed for check 3)\n")
for g in ["identity"] + X.G_LIST:
    row = []
    for s in SEEDS:
        tr = (g, D_MODEL, s) in TR; sh, dj = (g, "shared", D_MODEL, s) in J, (g, "disjoint", D_MODEL, s) in J
        ck = g != "identity" and os.path.exists(X.joint_paths(ROOT, g, "shared", D_MODEL, s)[1])
        row.append(f"s{s:<2d}: {'T' if tr else '.'} {'S' if sh else '.'}{'D' if dj else '.'} {'Y' if ck else '.'}   ")
    print(f"{g:13s} " + "".join(row))

T = translator, S/D = shared/disjoint joint, Y = shared checkpoint on disk (needed for check 3)

identity      s0 : T .. .   s7 : T .. .   s13: T .. .   
relabel_new   s0 : T SD Y   s7 : T .. .   s13: T .. .   
relabel_perm  s0 : T SD Y   s7 : T .. .   s13: T .. .   
marker        s0 : T SD Y   s7 : T .. .   s13: T .. .   
swap5         s0 : T SD Y   s7 : T .. .   s13: T .. .   
rev6          s0 : T SD Y   s7 : T .. .   s13: T .. .   
rev5          s0 : T SD Y   s7 : T .. .   s13: T .. .   
rev4          s0 : T SD Y   s7 : T .. .   s13: T .. .   
rev45         s0 : T SD Y   s7 : T .. .   s13: T .. .   
rev_all       s0 : T SD Y   s7 : T .. .   s13: T .. .   
perm_sym      s0 : T SD Y   s7 : T .. .   s13: T .. .   


## STEP 5. The gate (only the anchor language A', nothing about any rendering's sharing)

In [7]:
J, TR = load_all(ROOT)
Gr = gains(J, D_MODEL, SEEDS, use_floor=False)      # raw CE difference (confounded by grammar entropy)
Gn = gains(J, D_MODEL, SEEDS)                       # excess over the entropy floor (the one the checks use)
print("floors, nats/token:  shared grammar", round(floor_ce("shared", "A2", "rev6"), 4), "| disjoint A'", round(floor_ce("disjoint", "A2", "rev6"), 4), "(different grammars have different entropy)\n")
print(f"{'g':13s} {'seed':>4s} {'raw gain':>9s} {'corrected':>10s} {'SE':>7s} {'z':>6s}  {'excess shared':>13s} {'excess disjoint':>15s}   (all for the anchor language A')")
for (g, s), v in sorted(Gn.items()):
    print(f"{g:13s} {s:>4d} {Gr[(g, s)]['gain']['A2']:9.4f} {v['gain']['A2']:10.4f} {v['se']['A2']:7.4f} {v['gain']['A2'] / max(v['se']['A2'], 1e-9):6.1f}  {v['excess_shared']['A2']:13.4f} {v['excess_disjoint']['A2']:15.4f}")
ok, msg = gate_status(Gn)
print("\nGATE (floor-corrected):", msg, "->", "PASS" if ok else "FAIL")
print("PASS: sharing is visible at this d, carry on.  FAIL: untestable at this d (not negative); do not launch more runs at this d.")

floors, nats/token:  shared grammar 0.7645 | disjoint A' 0.7261 (different grammars have different entropy)

g             seed  raw gain  corrected      SE      z  excess shared excess disjoint   (all for the anchor language A')
marker           0   -0.0423    -0.0039  0.0014   -2.8         0.0085          0.0046
perm_sym         0   -0.0434    -0.0050  0.0014   -3.5         0.0102          0.0052
relabel_new      0   -0.0434    -0.0049  0.0014   -3.5         0.0106          0.0057
relabel_perm     0   -0.0441    -0.0057  0.0014   -4.0         0.0114          0.0058
rev4             0   -0.0427    -0.0042  0.0014   -3.0         0.0096          0.0053
rev45            0   -0.0452    -0.0068  0.0014   -4.9         0.0108          0.0041
rev5             0   -0.0445    -0.0061  0.0014   -4.3         0.0097          0.0037
rev6             0   -0.0428    -0.0044  0.0014   -3.2         0.0103          0.0059
rev_all          0   -0.0426    -0.0042  0.0014   -3.0         0.0103          0.0

## STEP 5a. (optional) Sanity check of the exact floor
Should reproduce exact floors 0.7645 / 0.7735 / 0.7261 / 0.7018 and show exact ≤ old everywhere.

In [8]:
# Optional sanity check, ~1.5 min: exact floor vs the old tree floor on the four grammars. Expected exact: shared 0.7645, disjoint A 0.7735, A' 0.7261, G 0.7018
import time; t0 = time.time()
def _tree_nll(node, rules): return 0.0 if isinstance(node, int) else math.log(len(rules[node[0]])) + sum(_tree_nll(c, rules) for c in node[1])
print(f"{'cond':9s}{'lang':5s}{'old(tree)':>11s}{'exact':>9s}{'old-exact':>11s}")
for cond, lang in [("shared", "A2"), ("disjoint", "A"), ("disjoint", "A2"), ("disjoint", "G")]:
    gram = X.make_grammars(cond)[lang]; tr = eval_trees(gram)
    old = float(np.mean([_tree_nll(t, gram) / len(X.render(t, {})) for t in tr])); ex = floor_ce(cond, lang, "rev6")
    print(f"{cond:9s}{lang:5s}{old:11.4f}{ex:9.4f}{old - ex:11.4f}   ({time.time() - t0:.0f}s)")
print("exact must never be above old (a string is at least as likely as one of its trees).")

cond     lang   old(tree)    exact  old-exact
shared   A2        0.7714   0.7645     0.0069   (2s)
disjoint A         0.7858   0.7735     0.0123   (3s)
disjoint A2        0.7379   0.7261     0.0118   (5s)
disjoint G         0.7158   0.7018     0.0139   (6s)
exact must never be above old (a string is at least as likely as one of its trees).


## STEP 5b. Record the amendment (once)
The instrument changed after the first gate read (grammar-entropy confound). This appends a dated note to the frozen file instead of silently changing it. Safe to re-run.

In [9]:
amend_prereg(ROOT)

amendment already recorded


In [10]:
# STEP 5c. Amendment 2 (once). Run this AFTER STEP 5 has confirmed that no excess is negative with the exact floor. Safe to re-run.
AMENDMENT_2 = ("AMENDMENT 2 - written {when}. Instrument change, recorded after the gate had been read once with the old floor (mean gain_A' about -0.0099).\n"
  "The floor used in Amendment 1 was nll(tree)/tokens and was called analytic. It is not: the model sees only the string, and a string can have several derivations, "
  "so the true floor is -log P(string)/tokens. It is now computed exactly with an inside algorithm (matrix products over spans) inside the notebook. "
  "On the 1000 evaluation trees the tree floor was too high by 0.0069 (shared grammar) and 0.0118 (disjoint A'). "
  "Validity check: no run may have a negative excess over this floor; if one does, this floor is also wrong. "
  "Gate thresholds (0.02 nats, 3 SE) and every check criterion are unchanged. Only the floor changed.")
if not RECORD_AMENDMENT_2: print("skipped (RECORD_AMENDMENT_2 is False)")
else: amend_prereg(ROOT, text=AMENDMENT_2)

skipped (RECORD_AMENDMENT_2 is False)


## STEP 6. Check 1: rev_all vs rev4 / rev45

In [11]:
r1 = check1(ROOT, d=D_MODEL, seeds=SEEDS)

=== CHECK 1: rev_all vs rev4 / rev45  (d=32) ===
GATE: mean gain_A' -0.0050 nats over 10 runs (SE of mean ~0.0004) -> FAIL
VERDICT: UNTESTABLE (gate). Not a negative result. Try smaller d.


## STEP 7. Check 2: does a(g) beat string distance?

In [12]:
r2 = check2(ROOT, d=D_MODEL, seeds=SEEDS)

=== CHECK 2: does a(g) beat string distance?  (d=32, seeds=[0, 7, 13]) ===
GATE: mean gain_A' -0.0050 nats over 10 runs (SE of mean ~0.0004) -> FAIL
VERDICT: UNTESTABLE (gate).


## STEP 8. Noise in a(g) across translator seeds

In [13]:
J, TR = load_all(ROOT)
seeds_a = [s for s in (0, 7, 13) if all((g, D_MODEL, s) in TR for g in X.G_LIST + ["identity"])]
A = {s: cost_a(TR, D_MODEL, [s]) for s in seeds_a}
gs = [g for g in X.G_LIST if all(g in A[s] for s in A)]
print("seeds with complete translators:", seeds_a)
if len(seeds_a) >= 2:
    print(f"{'g':13s}" + "".join(f"{'seed ' + str(s):>10s}" for s in A) + f"{'mean':>9s}{'sd':>9s}")
    for g in sorted(gs, key=lambda g: np.mean([A[s][g] for s in A])):
        v = [A[s][g] for s in A]; print(f"{g:13s}" + "".join(f"{x:10.4f}" for x in v) + f"{np.mean(v):9.4f}{np.std(v, ddof=1):9.4f}")
    for i in range(len(seeds_a)):
        for j in range(i + 1, len(seeds_a)):
            s1, s2 = seeds_a[i], seeds_a[j]; print(f"rank agreement seed {s1} vs {s2}: Spearman {spearman([A[s1][g] for g in gs], [A[s2][g] for g in gs]):+.2f}")
    print("Read it as: a gap between two renderings smaller than the sd is noise.")

seeds with complete translators: [0, 7, 13]
g                seed 0    seed 7   seed 13     mean       sd
relabel_perm     0.0033    0.0055    0.0080   0.0056   0.0024
relabel_new      0.0079    0.0082    0.0057   0.0072   0.0014
marker           0.0041    0.0107    0.0089   0.0079   0.0034
rev6             0.0085    0.0155    0.0111   0.0117   0.0035
rev_all          0.0237    0.0181    0.0148   0.0189   0.0045
swap5            0.0396    0.0362    0.0332   0.0363   0.0032
rev5             0.0438    0.0472    0.0435   0.0448   0.0021
rev45            0.1125    0.1336    0.1163   0.1208   0.0113
rev4             0.1387    0.1428    0.1455   0.1424   0.0034
perm_sym         0.2685    0.2762    0.2663   0.2703   0.0052
rank agreement seed 0 vs 7: Spearman +0.99
rank agreement seed 0 vs 13: Spearman +0.96
rank agreement seed 7 vs 13: Spearman +0.99
Read it as: a gap between two renderings smaller than the sd is noise.


## ⛔ STOP: everything below is guarded
GPU cells. They do nothing unless `RUN_GPU = True` in STEP 2. Run them only after STEP 5 shows GATE: PASS, and use `DEVICE = "cuda:0"` or another free GPU (check `nvidia-smi`).

In [14]:
# STEP 9a. depth smoke test on one rendering, a minute or so
if not RUN_GPU: print("skipped (RUN_GPU is False)")
else:
    run_one(ROOT, "rev4", D_MODEL, 0, DEVICE)
    r = json.load(open(f"{ROOT}/depth/rev4_d{D_MODEL}_s0.json"))["levels"]["4_centered"]
    print("T by layer :", np.round(r["T"], 2).tolist())
    print("A->A' acc  :", np.round(r["acc_AA2"], 2).tolist(), " (should be close to A->A)")
    print("A->A acc   :", np.round(r["acc_AA"], 2).tolist())
    print("g->g acc   :", np.round(r["acc_GG"], 2).tolist(), " chance:", round(r["chance"][0], 2))

skipped (RUN_GPU is False)


In [15]:
# STEP 9b. depth for every rendering and seed that has a shared checkpoint
if not RUN_GPU: print("skipped (RUN_GPU is False)")
else:
    for s in SEEDS:
        for g in X.G_LIST:
            if not os.path.exists(X.joint_paths(ROOT, g, "shared", D_MODEL, s)[1]): print("no checkpoint yet:", g, s); continue
            run_one(ROOT, g, D_MODEL, s, DEVICE)

skipped (RUN_GPU is False)


In [16]:
# STEP 9c. Check 3 verdict (CPU; reads whatever depth results exist)
if not os.path.isdir(f"{ROOT}/depth"): print("no depth results yet")
else: summarize(ROOT, d=D_MODEL, seeds=SEEDS)

no depth results yet


In [17]:
# STEP 10a. low-capacity pilot: does the gate pass at smaller d? 8 jobs; marker (cheap) and rev4 (costly), both conditions, d=16 and d=8
if not LAUNCH_PILOT: print("skipped (LAUNCH_PILOT is False)")
else:
    lines = []
    for dd in (16, 8):
        for g in ("marker", "rev4"):
            for cond in ("shared", "disjoint"):
                if not os.path.exists(X.joint_paths(ROOT, g, cond, dd, 0)[0]):
                    lines.append(f"{sys.executable} exp8_lib.py joint --g {g} --cond {cond} --seed 0 --d {dd} --steps 90000 --big_n 500000 --root {ROOT} --device {PILOT_DEVICE} > {ROOT}/logs/jt_{g}_{cond}_d{dd}_s0.txt 2>&1")
    open(f"{ROOT}/pilot_jobs.sh", "w").write("\n".join(lines) + "\n")
    p = subprocess.Popen(f"cat {ROOT}/pilot_jobs.sh | xargs -P 8 -I{{}} bash -c '{{}}'", shell=True, start_new_session=True, stdout=open(f"{ROOT}/pilot.log", "a"), stderr=subprocess.STDOUT)
    print(len(lines), "pilot jobs started, pid", p.pid)

skipped (LAUNCH_PILOT is False)


In [18]:
# STEP 10b. read the pilot gate (floor-corrected); change D_PILOT to look at d=8 or d=16
D_PILOT = 16
J, TR = load_all(ROOT); Gp = gains(J, D_PILOT, [0]); ok, msg = gate_status(Gp)
print(f"{'g':8s} {'corrected gain_A_prime':>22s} {'excess shared':>14s} {'excess disjoint':>16s}")
for (g, s), v in sorted(Gp.items()): print(f"{g:8s} {v['gain']['A2']:22.4f} {v['excess_shared']['A2']:14.4f} {v['excess_disjoint']['A2']:16.4f}")
print(f"\nPILOT GATE at d={D_PILOT}:", msg, "->", "PASS" if ok else "FAIL")

g        corrected gain_A_prime  excess shared  excess disjoint
marker                  -0.0000         0.0134           0.0133
rev4                    -0.0040         0.0164           0.0124

PILOT GATE at d=16: mean gain_A' -0.0020 nats over 2 runs (SE of mean ~0.0010) -> FAIL


In [19]:
# STEP 10c. start joint seeds 7 and 13 (about 2 hours per round); only after GATE: PASS
if not LAUNCH_JOINT_SEEDS: print("skipped (LAUNCH_JOINT_SEEDS is False)")
else:
    alive = subprocess.run("ps -eo args | grep exp8_queue.py | grep -v grep", shell=True, capture_output=True, text=True).stdout.strip()
    if alive: print("a scheduler is still running, not starting another:\n", alive)
    else:
        if os.path.exists(f"{ROOT}/scheduler.pid"): os.remove(f"{ROOT}/scheduler.pid")
        cmd = [sys.executable, "exp8_queue.py", "--root", ROOT, "--d", str(D_MODEL), "--seeds", "7", "13", "--devices", "cuda:2", "--max_par", "11", "--joint"]
        p = subprocess.Popen(cmd, stdout=open(f"{ROOT}/scheduler.log", "a"), stderr=subprocess.STDOUT, start_new_session=True); print("started, pid", p.pid)

skipped (LAUNCH_JOINT_SEEDS is False)


## STEP 10d. Full rendering grid at d=8 (after the pilot passed)
Uses `PILOT_DEVICE` from STEP 10a's cell; same resumable pattern, skips files that already exist.

In [20]:
# STEP 10d. d=8, seed 0, ALL renderings — first read on the full grid now that the gate is open at d=8.
# 20 jobs (10 renderings x 2 conditions); marker/rev4 already exist from the pilot and are skipped.
D_GRID, SEED_GRID = 8, 0
lines = []
for g in X.G_LIST:
    for cond in ("shared", "disjoint"):
        if not os.path.exists(X.joint_paths(ROOT, g, cond, D_GRID, SEED_GRID)[0]):
            lines.append(f"{sys.executable} exp8_lib.py joint --g {g} --cond {cond} --seed {SEED_GRID} --d {D_GRID} --steps 90000 --big_n 500000 --root {ROOT} --device {PILOT_DEVICE} > {ROOT}/logs/jt_{g}_{cond}_d{D_GRID}_s{SEED_GRID}.txt 2>&1")
if not lines:
    print("nothing to launch, all 20 already exist")
else:
    open(f"{ROOT}/grid_d8_jobs.sh", "w").write("\n".join(lines) + "\n")
    p = subprocess.Popen(f"cat {ROOT}/grid_d8_jobs.sh | xargs -P 8 -I{{}} bash -c '{{}}'", shell=True, start_new_session=True,
                          stdout=open(f"{ROOT}/grid_d8.log", "a"), stderr=subprocess.STDOUT)
    print(len(lines), "jobs started (", len(lines)//2, "renderings missing), pid", p.pid)

nothing to launch, all 20 already exist


## STEP 11. d=8, seed 0, full grid — first read (provisional: one seed, not yet the 3-seed criterion)

In [21]:
D_MODEL = 8
J, TR = load_all(ROOT); Gr = gains(J, D_MODEL, [0]); ok, msg = gate_status(Gr)
print(f"{'g':8s} {'corrected gain_A_prime':>22s} {'excess shared':>14s} {'excess disjoint':>16s}")
for (g, s), v in sorted(Gr.items()):
    print(f"{g:8s} {v['gain']['A2']:22.4f} {v['excess_shared']['A2']:14.4f} {v['excess_disjoint']['A2']:16.4f}")
print(f"\nGATE at d={D_MODEL}:", msg, "->", "PASS" if ok else "FAIL")

g        corrected gain_A_prime  excess shared  excess disjoint
marker                   0.0428         0.0625           0.1053
perm_sym                 0.0670         0.0762           0.1432
relabel_new                 0.0575         0.0595           0.1170
relabel_perm                 0.0755         0.0691           0.1446
rev4                     0.0593         0.0596           0.1190
rev45                    0.0485         0.0655           0.1140
rev5                     0.0418         0.0612           0.1030
rev6                     0.0533         0.0677           0.1210
rev_all                  0.0552         0.0776           0.1327
swap5                    0.0357         0.0584           0.0941

GATE at d=8: mean gain_A' 0.0537 nats over 10 runs (SE of mean ~0.0005) -> PASS


In [22]:
r1_d8 = check1(ROOT, d=D_MODEL, seeds=[0])

=== CHECK 1: rev_all vs rev4 / rev45  (d=8) ===
GATE: mean gain_A' 0.0537 nats over 10 runs (SE of mean ~0.0005) -> PASS

 seed s(rev_all)  s(rev4)  s(rev45)    diff   perm_sym s / rank among all
    0     -0.262    1.317    -0.609  -1.579   0.175 / 9 of 10

VERDICT: NO VERDICT YET - 1 usable seed(s), need 3. (Diffs so far: [-1.579].)


In [23]:
r2_d8 = check2(ROOT, d=D_MODEL, seeds=[0])

=== CHECK 2: does a(g) beat string distance?  (d=8, seeds=[0]) ===
GATE: mean gain_A' 0.0537 nats over 10 runs (SE of mean ~0.0005) -> PASS
VERDICT: NO VERDICT YET - 0 renderings with finite a and s, need 8.


## After the verdicts
- Report all three, failures included.
- Check 1 FAIL: a(g) loses its non-obvious argument; tell Anwoy before adding renderings.
- Check 2 INCONCLUSIVE: add rev3, rev345, rev2, rev23 and more seeds.
- Check 3 UNTESTABLE: read mean T and the shared-model gate.